# Interactive Prediction Form
Lokasyon bazli minute-risk tahmini ve grid isi haritasi uretimi.

In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'mine' else Path.cwd()
PIPE = ROOT / 'pipeline'
MODELS = ROOT / 'models'
if str(PIPE) not in sys.path:
    sys.path.insert(0, str(PIPE))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image

lat_w = widgets.Text(value='39.55.48N', description='Lat:')
lon_w = widgets.Text(value='32.51.00E', description='Lon:')
start_w = widgets.Text(value='2026-04-01T00:00:00Z', description='Start:')
end_w = widgets.Text(value='2026-04-03T00:00:00Z', description='End:')
topk_w = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Top-K:')

grid_time_w = widgets.Text(value='2026-04-01T00:00:00Z', description='Grid Time:')
latmin_w = widgets.Text(value='30.00.00N', description='Lat Min:')
latmax_w = widgets.Text(value='50.00.00N', description='Lat Max:')
lonmin_w = widgets.Text(value='20.00.00E', description='Lon Min:')
lonmax_w = widgets.Text(value='45.00.00E', description='Lon Max:')
step_w = widgets.FloatSlider(value=5.0, min=1.0, max=10.0, step=0.5, description='Step:')
grid_topk_w = widgets.IntSlider(value=50, min=10, max=200, step=10, description='Grid Top-K:')

run_pred_btn = widgets.Button(description='Run Location Prediction', button_style='success')
run_grid_btn = widgets.Button(description='Generate Grid Heatmap', button_style='info')
out = widgets.Output()

def on_pred(_):
    with out:
        out.clear_output()
        out_csv = MODELS / 'interactive_predictions.csv'
        cmd = (
            f"python3 {PIPE / 'predict_location_cli.py'} "
            f"--lat {lat_w.value} --lon {lon_w.value} --start {start_w.value} --end {end_w.value} "
            f"--top-k {topk_w.value} --out {out_csv}"
        )
        print(cmd)
        rc = os.system(cmd)
        if rc != 0:
            print('Prediction command failed')
            return
        df = pd.read_csv(out_csv)
        display(df[['time_utc','latitude_dms','longitude_dms','p_eq_m4_plus','p_eq_m4_plus_low95','p_eq_m4_plus_high95','pred_magnitude_if_event','pred_magnitude_q10','pred_magnitude_q90']].head(topk_w.value))

def on_grid(_):
    with out:
        out.clear_output()
        out_csv = MODELS / 'interactive_grid.csv'
        out_png = MODELS / 'interactive_grid.png'
        cmd = (
            f"python3 {PIPE / 'run_grid_pipeline.py'} --time {grid_time_w.value} "
            f"--lat-min {latmin_w.value} --lat-max {latmax_w.value} --lon-min {lonmin_w.value} --lon-max {lonmax_w.value} "
            f"--step {step_w.value} --top-k {grid_topk_w.value} --out-csv {out_csv} --out-png {out_png} "
            f"--title 'Interactive Grid Risk {grid_time_w.value}'"
        )
        print(cmd)
        rc = os.system(cmd)
        if rc != 0:
            print('Grid pipeline failed')
            return
        top_csv = out_csv.with_name(out_csv.stem + '_topk.csv')
        if top_csv.exists():
            top_df = pd.read_csv(top_csv)
            display(top_df[['latitude_dms','longitude_dms','p_eq_m4_plus','p_eq_m4_plus_low95','p_eq_m4_plus_high95','pred_magnitude_if_event']].head(20))
        if out_png.exists():
            display(Image(filename=str(out_png)))

run_pred_btn.on_click(on_pred)
run_grid_btn.on_click(on_grid)

display(widgets.VBox([
    widgets.HTML('<h3>Location Prediction</h3>'),
    lat_w, lon_w, start_w, end_w, topk_w, run_pred_btn,
    widgets.HTML('<h3>Grid Heatmap</h3>'),
    grid_time_w, latmin_w, latmax_w, lonmin_w, lonmax_w, step_w, grid_topk_w, run_grid_btn,
    out
]))